# 14_04_OpenTelemetry_で_RAG＋エージェ

コード：`ch14/blocks/14_04_OpenTelemetry_で_RAG＋エージェ.ipynb`

本文の断片は参照表示し、末尾のセルで前提を補う既存ランナーを実行します。関数・変数・外部処理が省略されているため、断片だけを単独実行しないでください。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import os
import pathlib
import sys
import zipfile

CODE = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents,
                        pathlib.Path.cwd() / 'gradient-attention-agent']
             if (p / 'run_all.py').is_file() and (p / 'common/blocks.py').is_file()), None)
if CODE is None:
    from google.colab import files
    uploaded = files.upload()  # gradient-attention-agent-code.zip を選ぶ
    if len(uploaded) != 1:
        raise ValueError('配布ZIPを1つ選んでください')
    dest = pathlib.Path.cwd().resolve()
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        for member in z.namelist():
            if not (dest / member).resolve().is_relative_to(dest / 'gradient-attention-agent'):
                raise ValueError('同梱のコード配布ZIPを選んでください')
        z.extractall(dest)
    CODE = dest / 'gradient-attention-agent'
    del uploaded
print('コード一式:', CODE)


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


## 本文の掲載コード（参照）


```python
def retrieve(query, k=50, k_final=5):
    with tracer.start_as_current_span("rag.retrieve") as sp:
        sp.set_attribute("rag.query.sha256", digest(query))
        with tracer.start_as_current_span("db.query") as s1:
            cands = vector_search(query, k=k)
            s1.set_attribute("db.system.name", "lancedb")
            s1.set_attribute("rag.top_k", k)
            s1.set_attribute("rag.hit_ids", [c.id for c in cands][:20])  # ID だけ
            s1.set_attribute("rag.top_score", cands[0].score if cands else 0.0)

        with tracer.start_as_current_span("rag.rerank") as s2:
            ranked = cross_encoder_rerank(query, cands)[:k_final]
            before = {c.id: i for i, c in enumerate(cands)}
            # ★ リランク前の順位を残す＝「正解を落としたか」が後から分かる
            s2.set_attribute("rag.rank_before", [before[c.id] for c in ranked])
            s2.set_attribute("rag.kept_ids", [c.id for c in ranked])
            s2.set_attribute("rag.min_score", min(c.score for c in ranked) if ranked else 0.0)

        sp.set_attribute("rag.returned", len(ranked))
        return ranked
```


## 前提を補って実行する

既存ランナーが関連する断片と必要なデータ・スタブを接続します。外部サービスを実行した証明や、掲載用AI応答の完全証跡にはなりません。


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, str(CODE / "run_all.py"), "--only", "14", "--name=otel", "--skip-docker"], cwd=CODE, check=True)
